# Bronze Layer

This process are build with Auto loader, implementing thetrigger(availableNow=True) option for execute this with serverless cluster, also contain the next features:
- If the tables are not created the process will create that, for avoid the schema conflicts.
- The tables are external in the bucket S3 location
- The schemalocation and checkpointlocation are in the bucket S3 too.
- Include the catalog like and external variable in the job

In [0]:

from pyspark.sql import functions as F

def load_bronze_layer(catalog, table, year, day, month):
    schemaLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/schemaLocation/{table}"
    input_path = f"s3://bakery-pos-bk-978430483777-us-east-2-an/bakery-pos/{year}/{month}/{day}/{table}"
    checkpointLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/checkpointLocation/{table}"
    tableLocation = f"s3://bakery-pos-bk-978430483777-us-east-2-an/devel_bronze_layer/{table}"
    
    files = dbutils.fs.ls(input_path)
    if not files:
        print(f"No files found in {input_path}")
        return

    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schemaLocation)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("multiLine", "true")
        .load(input_path)
        .select(
            "*",
            F.col("_metadata.file_name").alias("file_name"),
            F.col("_metadata.file_path").alias("file_path"),
            F.col("_metadata.file_modification_time").alias("file_modification_time"),
            F.current_timestamp().alias("processing_timestamp")
        )
    )

    raw_df.printSchema()

    query = (
        raw_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpointLocation)
        .option("path", tableLocation) 
        .trigger(availableNow=True)
        .table(f"{catalog}.bronze.{table}")
    )

    query.awaitTermination()

### _Execution_ bronze layer by each table in the schema

In [0]:
import datetime
import pytz

bogota_tz = pytz.timezone("America/Bogota")

year = datetime.datetime.now(tz=bogota_tz).strftime("%Y")
day = datetime.datetime.now(tz=bogota_tz).strftime("%d")
month = datetime.datetime.now(tz=bogota_tz).strftime("%m")

tables = [
    "account_move",
    "account_move_line",
    "pos_order",
    "pos_order_line",
    "pos_order_payment",
    "stock_move",
    "stock_picking"
]

for table in tables:
    try:
        print(f"Processing {table}...")
        load_bronze_layer("bakery_pos_develop", table, year, day, month)
        print(f"Successfully loaded {table}")
    except Exception as e:
        print(f"Error loading {table}: {str(e)}")
        continue

In [0]:
%sql

"""
DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_move;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.account_move_line;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_order;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_order_line;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.pos_order_payment;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.stock_move;
DROP TABLE IF EXISTS bakery_pos_develop.bronze.stock_picking;
"""

In [0]:
%sql

SELECT * FROM bakery_pos_develop.bronze.account_move LIMIT 10